In [ ]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-core \
    langchain-huggingface \
    langchain-chroma \
    chromadb \
    pypdf \
    sentence-transformers \
    google-genai \
    "requests==2.32.4" \
    "opentelemetry-api<=1.42.1" \
    "opentelemetry-sdk<=1.42.1"

In [ ]:
import os
import shutil
import traceback

from getpass import getpass
from google.colab import files

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from google import genai

print("✅ Imports successful!")

✅ Imports successful!


In [ ]:
# Clear any previous key
os.environ.pop("GEMINI_API_KEY", None)

print("=" * 60)
print("🔑 GEMINI API KEY")
print("=" * 60)

api_key = getpass("Enter your Gemini API key: ").strip()

if not api_key:
    raise ValueError("❌ API key cannot be empty.")

if not api_key.isascii():
    raise ValueError(
        "❌ API key contains an invalid character. "
        "Copy it again directly from Google AI Studio."
    )

os.environ["GEMINI_API_KEY"] = api_key

print("\n✅ Gemini API key loaded successfully!")

🔑 GEMINI API KEY
Enter your Gemini API key: ··········

✅ Gemini API key loaded successfully!


In [ ]:
GEMINI_MODEL = "gemini-3.6-flash"

print("Using Gemini model:", GEMINI_MODEL)

Using Gemini model: gemini-3.6-flash


In [ ]:
try:

    client = genai.Client(
        api_key=os.environ["GEMINI_API_KEY"]
    )

    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents="Reply with exactly: Gemini connection successful."
    )

    print("✅ GEMINI CONNECTION SUCCESSFUL!")
    print("\nGemini response:")
    print(response.text)

except Exception as e:

    print("❌ GEMINI CONNECTION FAILED")
    print("\nError type:", type(e).__name__)
    print("Error:", str(e))

✅ GEMINI CONNECTION SUCCESSFUL!

Gemini response:
Gemini connection successful.


In [ ]:
print("=" * 60)
print("📚 COLLEGE NOTES AI ASSISTANT")
print("=" * 60)

print("\nUpload your college PDF notes:")

uploaded = files.upload()

pdf_files = [
    filename
    for filename in uploaded.keys()
    if filename.lower().endswith(".pdf")
]

if not pdf_files:
    raise ValueError("❌ No PDF files uploaded.")

print("\n✅ Uploaded PDFs:")

for pdf in pdf_files:
    print("📄", pdf)

print("\nTotal PDFs:", len(pdf_files))

📚 COLLEGE NOTES AI ASSISTANT

Upload your college PDF notes:


Saving sem520.pdf to sem520.pdf

✅ Uploaded PDFs:
📄 sem520.pdf

Total PDFs: 1


In [ ]:
documents = []

for pdf_file in pdf_files:

    print(f"\nLoading: {pdf_file}")

    try:

        loader = PyPDFLoader(pdf_file)
        pdf_docs = loader.load()

        for doc in pdf_docs:
            doc.metadata["source_file"] = pdf_file

        documents.extend(pdf_docs)

        print(
            f"✅ Loaded {len(pdf_docs)} pages"
        )

    except Exception as e:

        print(
            f"❌ Error loading {pdf_file}: {e}"
        )

print("\n" + "=" * 60)
print("PDF LOADING COMPLETE")
print("=" * 60)

print("Total pages:", len(documents))


Loading: sem520.pdf
✅ Loaded 22 pages

PDF LOADING COMPLETE
Total pages: 22


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(
    documents
)

print("✅ Documents split successfully!")
print("Total chunks:", len(chunks))

✅ Documents split successfully!
Total chunks: 52


In [ ]:
print("Loading embedding model...")

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("✅ Embedding model loaded!")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded!


In [ ]:
if os.path.exists("./college_notes_db"):
    shutil.rmtree("./college_notes_db")

print("Creating Chroma vector database...")

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="college_notes",
    persist_directory="./college_notes_db"
)

print("✅ Chroma vector database created!")

Creating Chroma vector database...
✅ Chroma vector database created!


In [ ]:
retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

print("✅ Retriever ready!")

✅ Retriever ready!


In [ ]:
SYSTEM_PROMPT = """
You are a College Notes AI Assistant.

Answer the student's question ONLY using the
information contained in the provided CONTEXT.

STRICT RULES:

1. Use only the provided context.
2. Do not use outside knowledge.
3. Do not invent or hallucinate information.
4. If the answer cannot be found in the context,
   respond exactly:

"I could not find the answer to this question
in the uploaded documents."

5. Give a clear, student-friendly explanation.
6. Keep the answer relevant to the question.
7. Do not add unsupported facts.

CONTEXT:
{context}

STUDENT QUESTION:
{question}

ANSWER:
"""

print("✅ RAG prompt ready!")

✅ RAG prompt ready!


In [ ]:
def ask_question(question):

    question = str(question).strip()

    if not question:
        return "Please enter a question."

    # 1. Retrieve relevant chunks
    retrieved_docs = retriever.invoke(question)

    if not retrieved_docs:
        return (
            "I could not find the answer to this question "
            "in the uploaded documents."
        )

    # 2. Build context
    context_parts = []

    for doc in retrieved_docs:

        source = str(
            doc.metadata.get(
                "source_file",
                "Unknown source"
            )
        )

        page = doc.metadata.get(
            "page",
            "Unknown page"
        )

        if isinstance(page, int):
            page += 1

        content = str(doc.page_content)

        context_parts.append(
            f"""
Source: {source}
Page: {page}

Content:
{content}
"""
        )

    context = "\n\n--------------------\n\n".join(
        context_parts
    )

    # 3. Create final prompt
    final_prompt = SYSTEM_PROMPT.format(
        context=context,
        question=question
    )

    # 4. Ask Gemini
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=final_prompt
    )

    return response.text

In [ ]:
question = "What is a deadlock?"

print("=" * 60)
print("QUESTION")
print("=" * 60)
print(question)

print("\n" + "=" * 60)
print("ANSWER")
print("=" * 60)

try:

    answer = ask_question(question)

    print(answer)

except Exception:

    print("❌ Error:")
    traceback.print_exc()

QUESTION
What is a deadlock?

ANSWER
I could not find the answer to this question in the uploaded documents.


In [ ]:
question = "What are the four necessary conditions for deadlock?"

print("QUESTION:")
print(question)

print("\nANSWER:")

try:
    print(ask_question(question))

except Exception:
    traceback.print_exc()

QUESTION:
What are the four necessary conditions for deadlock?

ANSWER:
I could not find the answer to this question in the uploaded documents.


In [ ]:
question = "Explain TCP vs UDP."

print("QUESTION:")
print(question)

print("\nANSWER:")

try:
    print(ask_question(question))

except Exception:
    traceback.print_exc()

QUESTION:
Explain TCP vs UDP.

ANSWER:
I could not find the answer to this question in the uploaded documents.


In [ ]:
question = "What is a binary search tree?"

print("QUESTION:")
print(question)

print("\nANSWER:")

try:
    print(ask_question(question))

except Exception:
    traceback.print_exc()

QUESTION:
What is a binary search tree?

ANSWER:
I could not find the answer to this question in the uploaded documents.


In [ ]:
question = "What is the history of the Taj Mahal?"

print("QUESTION:")
print(question)

print("\nANSWER:")

try:
    print(ask_question(question))

except Exception:
    traceback.print_exc()

QUESTION:
What is the history of the Taj Mahal?

ANSWER:
I could not find the answer to this question in the uploaded documents.


In [ ]:
print("\n" + "=" * 70)
print("📚 COLLEGE NOTES AI ASSISTANT")
print("=" * 70)

print("""
Your college PDFs are now loaded into the knowledge base.

Ask anything related to your uploaded notes.

Type 'exit' to stop.
""")

while True:

    try:

        question = input("\nYou: ").strip()

        if question.lower() == "exit":

            print("\nAssistant: Goodbye! 👋")
            break

        if not question:

            print(
                "Assistant: Please enter a question."
            )
            continue

        print("\nAssistant:")

        answer = ask_question(question)

        print(answer)

        print("\n" + "-" * 70)

    except KeyboardInterrupt:

        print("\n\nAssistant: Chat ended.")
        break

    except Exception:

        print("\n❌ Error:")
        traceback.print_exc()